# Faza I – Para-procesimi i të dhënave

**Prokurimi publik në Shqipëri, 2016–2025** · Grupi 13 · Përgatitja dhe vizualizimi i të dhënave

Ky notebook tregon hap pas hapi punën e Fazës I, sipas tri grupeve të kërkesave të profesorit:

- **a)** Mbledhja e të dhënave, definimi i tipeve, kualiteti i të dhënave
- **b)** Integrimi, agregimi, mostrimi, pastrimi, identifikimi dhe strategjia e trajtimit për vlerat e zbrazëta
- **c)** Reduktimi i dimensionit, zgjedhja e nën-bashkësisë së vetive, krijimi i vetive, diskretizimi dhe binarizimi, transformimi

Notebook-u **nuk e rishkruan logjikën**: thërret funksionet e kodit në `src/`, të ndara në tri pjesë:

| Pjesa | Anëtari | Kodi | Dokumentimi |
|---|---|---|---|
| Vlerat dhe fituesit | Marigona Hamiti | `src/values_winners/` | `docs/values_winners.md` |
| Koha dhe integrimi | Semine Muharremi | `src/time_integration/` | `docs/time_integration.md` |
| Procedura dhe konkurrenca | Tringa Baftiu | `src/procedure_competition/` | `docs/procedure_competition.md` |

**Para ekzekutimit:** 10 skedarët CSV të APP-së (2016–2025) duhet të jenë te `data/raw/` (shih README). Ekzekutimi i plotë zgjat rreth 2 minuta.

In [1]:
import os
import sys
from pathlib import Path

# Notebook-u ekzekutohet nga rrënja e repository-t, që të gjenden src/ dhe data/
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

DATA = Path("data/raw")

---
# a) Mbledhja, tipet dhe kualiteti

## 1. Mbledhja e të dhënave

Burimi: **Agjencia e Prokurimit Publik (APP)**, [Eksportimi i procedurave të publikuara](https://www.app.gov.al/eksportimi-i-procedurave-te-publikuara/). Një skedar CSV për çdo vit, 21 kolona. Viti 2026 përjashtohet, sepse në datën e eksportit nuk ishte i plotë. Të gjitha kolonat lexohen si tekst, që të ruhet formati origjinal.

**Objekti** i të dhënave është një procedurë prokurimi (tender): çdo rresht është një procedurë. Dataset-i është i tipit **record data**, me komponent kohor (datat) dhe kategori (autoriteti, procedura).

In [2]:
from src.common.data_loading import read_years

raw = read_years(DATA)
print(f"{len(raw):,} procedura x {raw.shape[1] - 1} kolona (+ kolona 'year')")
raw.groupby("year").size().rename("procedura").to_frame().T

277,764 procedura x 21 kolona (+ kolona 'year')


year,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
procedura,33375,31769,33880,28313,23670,22653,26034,25137,27466,25467


In [3]:
# Rreshtat janë observimet (procedurat), kolonat janë variablat
raw.head(3)

,Autoriteti_kontraktues,Numri_i_references,Objekti_i_prokurimit,Lloji_i_procedures,Tipi_i_kontrates,Lloji_i_marreveshjes_kuader,Fondi_limit,Data_e_publikimit,Data_e_hapjes,Data_e_mbylljes,Anulluar,Arsyeja_e_anullimit,Pezulluar,Fituesi,NIPT_i_fituesit,Vlera_e_fituesit,Vlera_e_fituesit_ne_lidhjen_e_kontrates,Lidhja_e_kontrates_me_TVSH,Numri_i_ofertave_te_dorezuara,Numri_i_ofertave_te_kualifikuara,Kodet_CPV,year
0,Instituti i Integrimit te Perndjekurve Politike,REF-31677-04-04-2016,Blerje Pajisje Elektronike,Small Value,Supplies,NaN,"741,600.00",04.04.2016,04.04.2016,05.04.2016,Po,Pёr shkak se nga tё gjithё operatorёt e klasifikuar tё renditur nё 5(pesё) v...,Jo,KREATX,L12121023B,"390,000.00",NaN,Jo,26,26,NaN,2016
1,Instituti i Integrimit te Perndjekurve Politike,REF-51109-09-27-2016,"""Furnizim vendosje Kondicionere, Televizor""",Small Value,Supplies,NaN,"208,330.00",27.09.2016,27.09.2016,28.09.2016,Po,"Sipas Udhezimit nr. 5 dt 27.01.2015 ""Per zhvillimin e procedurave te prokuri...",Jo,ALTEC SHPK,K57923807W,"185,000.00",NaN,Jo,23,23,NaN,2016
2,Zyra e Prokurorit te Pergjithshem,REF-43875-07-01-2016,"Blerje automjetesh, për Prokurorinë e Përgjithshme",Open Local,Supplies,NaN,"25,000,000.00",04.07.2016,04.07.2016,05.10.2016,Jo,NaN,Jo,PORSCHE ALBANIA SH.P.K,K51914014P,"29,649,600.00","29,649,600.00",Jo,2,1,NaN,2016


In [4]:
# Tipi që Pandas i ka dhënë çdo kolone: të gjitha janë tekst, sepse u lexuan si tekst
raw.dtypes.value_counts()

str      21
int64     1
Name: count, dtype: int64

### Kontrolli: a përputhet tipi i kolonës me tipin e vlerave?

Për çdo kolonë tekstuale kontrollojmë sa përqind e vlerave mund të kthehen në **numër** (pasi hiqet presja e mijësheve, `"741,600.00"` → `741600.00`) ose në **datë** (`DD.MM.YYYY`). Nëse pothuajse të gjitha vlerat kthehen, kolona në të vërtetë është numerike ose datë dhe duhet konvertuar te pastrimi.

In [5]:
text_cols = [c for c in raw.columns if c != "year"]
scan = []
for col in text_cols:
    values = raw[col].dropna().str.strip()
    as_number = pd.to_numeric(values.str.replace(",", "", regex=False), errors="coerce")
    as_date = pd.to_datetime(values, format="%d.%m.%Y", errors="coerce")
    scan.append({
        "Kolona": col,
        "Vlera": len(values),
        "% numër": round(as_number.notna().mean() * 100, 2),
        "% datë": round(as_date.notna().mean() * 100, 2),
    })
scan = pd.DataFrame(scan)
scan["Tipi i vërtetë"] = scan.apply(
    lambda r: "numerik" if r["% numër"] > 95 else ("datë" if r["% datë"] > 95 else "tekst / kategorik"), axis=1)
scan

,Kolona,Vlera,% numër,% datë,Tipi i vërtetë
0,Autoriteti_kontraktues,277764,0.00,0.0,tekst / kategorik
1,Numri_i_references,277764,0.00,0.0,tekst / kategorik
2,Objekti_i_prokurimit,277761,0.01,0.0,tekst / kategorik
3,Lloji_i_procedures,277764,0.00,0.0,tekst / kategorik
4,Tipi_i_kontrates,277764,0.00,0.0,tekst / kategorik
5,Lloji_i_marreveshjes_kuader,19424,0.00,0.0,tekst / kategorik
6,Fondi_limit,252200,100.00,0.0,numerik
7,Data_e_publikimit,277764,0.00,100.0,datë
8,Data_e_hapjes,277595,0.00,100.0,datë
9,Data_e_mbylljes,276672,0.00,100.0,datë


## 2. Definimi i tipeve të të dhënave

Çdo pjesë ka përcaktuar tipin e kolonave të veta: kategorik (nominal, ordinal, binar, tekst) ose numerik (interval, ratio), diskret ose i vazhdueshëm. Kolona `csv_type` tregon që **të gjitha** kolonat ruhen si tekst në CSV, pra tipi real duhet vendosur në pastrim.

**Shkallët e matjes:** nominale (p.sh. autoriteti, fituesi), ordinale (p.sh. kategoria e fondit), **intervale** (datat: diferenca mes dy datave ka kuptim, por nuk ka zero absolute) dhe raporti / ratio (shumat në lekë, numri i ofertave: kanë zero absolute).

In [6]:
from src.common.data_types import describe_types
from src.procedure_competition.columns import COLUMNS as PC_COLUMNS
from src.time_integration.columns import COLUMNS as TI_COLUMNS
from src.values_winners.columns import COLUMNS as VW_COLUMNS

PARTS = {
    "Vlerat dhe fituesit": VW_COLUMNS,
    "Koha dhe integrimi": {c: t for c, t in TI_COLUMNS.items() if c in raw},
    "Procedura dhe konkurrenca": PC_COLUMNS,
}
types = pd.concat([describe_types(raw, cols).assign(pjesa=name) for name, cols in PARTS.items()],
                  ignore_index=True)
types[["pjesa", "column", "category", "type", "nature", "csv_type", "unique_values"]]

,pjesa,column,category,type,nature,csv_type,unique_values
0,Vlerat dhe fituesit,Fondi_limit,Numeric,Ratio,Continuous,text,102064
1,Vlerat dhe fituesit,Vlera_e_fituesit,Numeric,Ratio,Continuous,text,70706
2,Vlerat dhe fituesit,Vlera_e_fituesit_ne_lidhjen_e_kontrates,Numeric,Ratio,Continuous,text,37018
3,Vlerat dhe fituesit,Lidhja_e_kontrates_me_TVSH,Categorical,Binary (Po/Jo),Discrete,text,2
4,Vlerat dhe fituesit,Fituesi,Categorical,Nominal,Discrete,text,8269
5,Vlerat dhe fituesit,NIPT_i_fituesit,Categorical,Nominal (identifier),Discrete,text,8305
6,Koha dhe integrimi,Data_e_publikimit,Categorical,Datetime,Discrete,text,3320
7,Koha dhe integrimi,Data_e_hapjes,Categorical,Datetime,Discrete,text,3353
8,Koha dhe integrimi,Data_e_mbylljes,Categorical,Datetime,Discrete,text,3193
9,Koha dhe integrimi,Numri_i_references,Numeric,Count,Discrete,text,277757


## 3. Kualiteti i të dhënave

Në Fazën I problemet **vetëm identifikohen dhe dokumentohen**; asnjë rresht nuk ndryshohet këtu. Çdo pjesë kontrollon kolonat e veta sipas dimensioneve të leksionit.

In [7]:
# Vlerat që mungojnë në të dhënat bruto, me përqindje
missing = pd.DataFrame({
    "Vlerat_munguese": raw.isna().sum(),
    "Perqindja_%": (raw.isna().mean() * 100).round(2),
})
missing[missing["Vlerat_munguese"] > 0].sort_values("Perqindja_%", ascending=False)

,Vlerat_munguese,Perqindja_%
Lloji_i_marreveshjes_kuader,258340,93.01
Vlera_e_fituesit_ne_lidhjen_e_kontrates,226259,81.46
Arsyeja_e_anullimit,205861,74.11
Kodet_CPV,61688,22.21
Vlera_e_fituesit,57382,20.66
Fituesi,53894,19.40
NIPT_i_fituesit,53880,19.40
Fondi_limit,25564,9.20
Data_e_mbylljes,1092,0.39
Data_e_hapjes,169,0.06


In [8]:
# Duplikatet: rreshta krejtësisht të njëjtë, dhe numra reference të përsëritur
print("Rreshta të dyfishuar (të 21 kolonat):", raw.drop(columns="year").duplicated().sum())
ref_dupes = raw[raw["Numri_i_references"].duplicated(keep=False)]
print("Procedura me referencë të përsëritur:", len(ref_dupes), "| referenca:", ref_dupes["Numri_i_references"].nunique())
ref_dupes.sort_values("Numri_i_references")[["Numri_i_references", "year", "Autoriteti_kontraktues", "Fituesi"]].head(6)

Rreshta të dyfishuar (të 21 kolonat): 0
Procedura me referencë të përsëritur: 14 | referenca: 7


,Numri_i_references,year,Autoriteti_kontraktues,Fituesi
236579,REF-20571-09-19-2024,2024,Drejtoria e Sherbimit Spitalor Skrapar,NOART
244044,REF-20571-09-19-2024,2024,Agjencia Kombetare e Planifikimit te Territorit,INFOSOFT OFFICE
21645,REF-22877-02-04-2016,2016,Klubi Sportiv Albpetrol Patos,ERJON BEGAJ
32364,REF-22877-02-04-2016,2016,Spitali Obsetrik Gjinekologjik Mbreteresha Geraldine,SUPPORT - 07 SH.P.K
10629,REF-35545-04-27-2016,2016,Instituti i Mjekesise Ligjore,MOBILERI CURRI
29956,REF-35545-04-27-2016,2016,Hekurudha Shqiptare sh.a,TECH INVEST


Nuk ka rreshta të dyfishuar, prandaj `drop_duplicates()` nuk heq asgjë. Referencat e përsëritura i përkasin autoriteteve të ndryshme, pra janë procedura të ndryshme dhe **nuk fshihen**; një procedurë identifikohet nga referenca **bashkë me** autoritetin.

Më poshtë, kontrollet e plota të çdo pjese, sipas dimensioneve të kualitetit.

In [9]:
from src.common.quality import run_checks
from src.procedure_competition.prepare import quality_report
from src.time_integration.quality import build_checks as time_checks
from src.values_winners.quality import build_checks as value_checks

quality = pd.concat([
    run_checks(raw, value_checks(raw)).assign(pjesa="Vlerat dhe fituesit"),
    time_checks(raw).assign(pjesa="Koha dhe integrimi"),  # e kthen tabelën të gatshme
    quality_report(raw[list(PC_COLUMNS)]).assign(pjesa="Procedura dhe konkurrenca"),
], ignore_index=True)

print(f"{len(quality)} kontrolle, {int((quality['rows'] > 0).sum())} me probleme")
quality.groupby(["pjesa", "dimension"]).size().unstack(fill_value=0)

76 kontrolle, 42 me probleme


dimension,Accuracy,Completeness,Consistency,Format,Uniqueness
pjesa,,,,,
Koha dhe integrimi,7,4,0,6,0
Procedura dhe konkurrenca,2,12,2,13,1
Vlerat dhe fituesit,8,6,4,8,3


In [10]:
# Problemet kryesore: kontrollet me më shumë rreshta të prekur
quality[quality["rows"] > 0].sort_values("rows", ascending=False).head(15)[
    ["pjesa", "dimension", "column", "check", "rows", "pct_of_base"]]

,pjesa,dimension,column,check,rows,pct_of_base
50,Procedura dhe konkurrenca,Completeness,Lloji_i_marreveshjes_kuader,Missing or blank value,258340,93.01
5,Vlerat dhe fituesit,Completeness,Vlera_e_fituesit_ne_lidhjen_e_kontrates,Missing value,226259,81.46
64,Procedura dhe konkurrenca,Format,Kodet_CPV,Leading/trailing or repeated whitespace,216076,100.00
52,Procedura dhe konkurrenca,Completeness,Arsyeja_e_anullimit,Missing or blank value,205861,74.11
75,Procedura dhe konkurrenca,Format,Kodet_CPV,More than one CPV code in cell,61968,28.68
56,Procedura dhe konkurrenca,Completeness,Kodet_CPV,Missing or blank value,61688,22.21
4,Vlerat dhe fituesit,Completeness,Vlera_e_fituesit,Missing value,57382,20.66
7,Vlerat dhe fituesit,Completeness,Fituesi,Missing value,53894,19.40
8,Vlerat dhe fituesit,Completeness,NIPT_i_fituesit,Missing value,53880,19.40
59,Procedura dhe konkurrenca,Format,Objekti_i_prokurimit,Leading/trailing or repeated whitespace,37462,13.49


**Dimensionet e tjera të leksionit:**

- **Believability (besueshmëria):** burimi është institucion shtetëror (APP), i detyruar me ligj të publikojë procedurat; nuk ka ndërhyrje manuale në skedarë.
- **Timeliness (aktualiteti):** eksporti është i tetorit 2026 dhe mbulon 2016–2025; viti 2026 përjashtohet sepse ishte i paplotë. Disa fusha plotësohen vetëm nga një vit i caktuar (p.sh. TVSH nga 2018), gjë që duhet mbajtur parasysh kur krahasohen vitet.

---
# b) Pastrimi, vlerat e zbrazëta, integrimi, agregimi dhe mostrimi

## 4. Pastrimi

Pastrimi rregullon formatin dhe konsistencën **pa fshirë asnjë rresht**. Vlerat logjikisht të pamundura (p.sh. vlera e fituesit mbi fondin) nuk ndryshohen: ato i takojnë Pjesës II.

In [11]:
from src.values_winners.cleaning import clean as clean_values, cleaning_summary as value_cleaning

values = clean_values(raw)
value_cleaning(raw, values)

,column,step,before,after
0,Fondi_limit,Text amount -> number,"text (252,200 values)","float64 (252,200 values)"
1,Vlera_e_fituesit,Text amount -> number,"text (220,382 values)","float64 (220,382 values)"
2,Vlera_e_fituesit_ne_lidhjen_e_kontrates,Text amount -> number,"text (51,505 values)","float64 (51,505 values)"
3,Lidhja_e_kontrates_me_TVSH,Po/Jo -> boolean,"{'Jo': 251197, 'Po': 26567}","{'False': np.int64(251197), 'True': np.int64(26567)}"
4,NIPT_i_fituesit,"Trim whitespace/newlines, upper case","8,305 distinct","7,444 distinct"
5,Fituesi,"Trim whitespace/newlines, upper case, one name per NIPT","8,269 distinct","7,403 distinct"


In [12]:
from src.time_integration.cleaning import clean as clean_dates, cleaning_summary as date_cleaning, date_order_checks
from src.time_integration.missing_values import fill_missing_dates, parse_dates

dates_raw = parse_dates(raw[[c for c in TI_COLUMNS if c in raw] + ["year"]])
dates = clean_dates(dates_raw)
display(date_cleaning(dates))

# Renditja e datave para dhe pas pastrimit
pd.DataFrame({"para": date_order_checks(dates_raw), "pas": date_order_checks(fill_missing_dates(dates))})

,column,problem,action,rows
0,Data_e_publikimit,"Opening before publication, after registration",Publication date moved back to the opening date,7110
1,Data_e_hapjes,Opening before registration or after closing,"Set to missing, then filled (publication + median delay)",65
2,Data_e_mbylljes,Closing before registration or after the export date (19.10.2026),"Set to missing, then filled (opening + median duration)",6


,para,pas
Publication before registration,0,0
Opening before publication,7170,0
Closing before opening,5,0
Closing before publication,245,0
Closing after export date,1,0
Tender duration > 365 days,5,4


In [13]:
from src.procedure_competition.prepare import clean_and_engineer

procedures = clean_and_engineer(raw[[*PC_COLUMNS, "year"]])
after = quality_report(procedures, cleaned=True)
after[after["dimension"] == "Format"][["column", "check", "rows"]]

,column,check,rows
12,Autoriteti_kontraktues,Leading/trailing or repeated whitespace,0
13,Objekti_i_prokurimit,Leading/trailing or repeated whitespace,0
14,Lloji_i_procedures,Leading/trailing or repeated whitespace,0
15,Tipi_i_kontrates,Leading/trailing or repeated whitespace,0
16,Lloji_i_marreveshjes_kuader,Leading/trailing or repeated whitespace,0
17,Arsyeja_e_anullimit,Leading/trailing or repeated whitespace,0
18,Kodet_CPV,Leading/trailing or repeated whitespace,0
19,Anulluar,Value outside boolean vocabulary,0
20,Pezulluar,Value outside boolean vocabulary,0
21,Numri_i_ofertave_te_dorezuara,Non-numeric value,0


## 5. Vlerat e zbrazëta: identifikimi dhe strategjia

Si në leksion, **të dhënat që mungojnë nuk fshihen**. Fondet 0 ose ≤ 1 lek janë **mungesa të maskuara** (vlera të paracaktuara kur fondi nuk publikohet) dhe shënohen si mungesë.

In [14]:
from src.common.missing import compare_numeric_strategies, missing_summary
from src.values_winners.missing_values import mark_disguised_missing

values_m = mark_disguised_missing(values)
missing_summary(values, VW_COLUMNS).merge(
    missing_summary(values_m, VW_COLUMNS), on="column", suffixes=("_bruto", "_me_te_maskuarat"))

,column,missing_bruto,missing_pct_bruto,missing_me_te_maskuarat,missing_pct_me_te_maskuarat
0,Fondi_limit,25564,9.20,42287,15.22
1,Vlera_e_fituesit,57382,20.66,57382,20.66
2,Vlera_e_fituesit_ne_lidhjen_e_kontrates,226259,81.46,226259,81.46
3,Lidhja_e_kontrates_me_TVSH,0,0.00,0,0.00
4,Fituesi,53894,19.40,53894,19.40
5,NIPT_i_fituesit,53880,19.40,53880,19.40


In [15]:
# Krahasimi i strategjive për fondin: fshirja humb rreshta, imputimi shpik vlera
compare_numeric_strategies(values_m["Fondi_limit"], raw["Lloji_i_procedures"])

,strategy,rows_kept,still_missing,median,mean
0,Original (with NaN),277764,42287,390000.0,9150130.36
1,Deletion (drop rows),235477,0,390000.0,9150130.36
2,Median imputation,277764,0,390000.0,7816481.53
3,Median by group,277764,0,263277.0,7789468.91
4,Keep NaN + missing flag,277764,42287,390000.0,9150130.36


**Strategjitë e zgjedhura:**
- **Vlerat dhe fituesit:** `NaN` + flag, pa imputim. 85% e procedurave pa fitues janë të anuluara dhe 83.5% e blerjeve të biletave nuk kanë fond, pra mungesat kanë shpjegim (MAR) dhe një vlerë e shpikur do të prishte analizën.
- **Koha:** datat e hapjes/mbylljes që mungojnë plotësohen me datën paraprake + medianën e vonesës së procedurave të të njëjtit lloj në të njëjtin vit (strategjia më e saktë nga 4 të krahasuara), me flag `*_imputed` (mungesat janë pak: 0.06% dhe 0.39%).
- **Procedura dhe konkurrenca:** mungesat ruhen si mungesa; nuk imputohen kategori të hamendësuara.

In [16]:
from src.time_integration.missing_values import TIME_COLS, imputation_report

imputation_report(fill_missing_dates(dates), missing_summary(dates, TIME_COLS))

,column,missing_before,missing_pct_before,missing_after,missing_pct_after,imputed
0,Data_e_publikimit,0,0.0,0,0.0,0
1,Data_e_hapjes,0,0.0,0,0.0,0
2,Data_e_mbylljes,0,0.0,0,0.0,0
3,Numri_i_references,0,0.0,0,0.0,0


In [17]:
from src.procedure_competition.prepare import missing_value_report

missing_value_report(raw[list(PC_COLUMNS)], procedures)

,column,missing_before,pct_before,missing_after,pct_after,treatment
0,Autoriteti_kontraktues,0,0.00,0,0.00,Whitespace normalized; blank values retained as missing
1,Objekti_i_prokurimit,3,0.00,3,0.00,Whitespace normalized; blank values retained as missing
2,Lloji_i_procedures,0,0.00,0,0.00,Whitespace normalized; blank values retained as missing
3,Tipi_i_kontrates,0,0.00,0,0.00,Whitespace normalized; blank values retained as missing
4,Lloji_i_marreveshjes_kuader,258340,93.01,258340,93.01,Retained as missing; no reliable source value to impute
5,Anulluar,0,0.00,0,0.00,Mapped Po/Jo to boolean; invalid or blank values retained as missing
6,Arsyeja_e_anullimit,205861,74.11,205861,74.11,Retained as missing; only applicable to cancelled procedures
7,Pezulluar,0,0.00,0,0.00,Mapped Po/Jo to boolean; invalid or blank values retained as missing
8,Numri_i_ofertave_te_dorezuara,0,0.00,0,0.00,Converted to nullable integer; missing values retained
9,Numri_i_ofertave_te_kualifikuara,0,0.00,0,0.00,Converted to nullable integer; missing values retained


## 6. Integrimi

Tri pjesët përgatiten nga i njëjti lexim dhe bashkohen **një-me-një** mbi çelësin `Numri_i_references` + `Autoriteti_kontraktues` (referenca vetëm nuk është unike: 7 referenca përdoren nga dy procedura). Integrohen edhe dy burime të jashtme: **datat e zgjedhjeve (KQZ)** dhe **ndarjet CPV** (Rregullorja EC 213/2008).

In [ ]:
from src.time_integration.integrated_dataset import column_sources, integrate, integration_summary, prepare_parts

parts = prepare_parts(raw)
integrated = integrate(parts)
integration_summary(raw, integrated, column_sources(integrated, parts))

## 7. Agregimi

Shumat përmblidhen me **medianë**: shuma dhe mesatarja do të dominoheshin nga vlerat e pamundura (deri në 35 mijë miliardë lekë) që trajtohen në Pjesën II. Interpretimi i dallimeve bëhet në Pjesën II.

In [ ]:
from src.values_winners.aggregation import by_group, concentration, winners

by_group(integrated, "year")

In [ ]:
by_group(integrated, "afersia_me_zgjedhjet")

In [ ]:
# Përqendrimi i tregut sipas numrit të kontratave
concentration(winners(integrated))

### Agregimi kohor me `resample`

Data e publikimit vendoset si indeks, që Pandas të mund të grupojë sipas periudhave kohore. Krahasohet niveli i agregimit: sa më e gjatë periudha, aq më pak vlera dhe aq më e qetë seria. Viti këtu është viti i **publikimit**, jo i skedarit: disa procedura publikohen pas vitit të skedarit (p.sh. regjistruar në dhjetor, publikuar në janar).

In [ ]:
by_date = integrated.set_index("Data_e_publikimit").sort_index()

# 223 procedura kanë datë publikimi pas vitit të skedarit (deri në shtator 2026);
# agregimi kohor kufizohet te periudha e projektit 2016-2025
outside = by_date.loc["2026":]
print(f"Procedura të publikuara pas 2025 (lihen jashtë agregimit kohor): {len(outside)}")
by_date = by_date.loc["2016":"2025"]

monthly = by_date.resample("MS").agg(procedura=("Fondi_limit", "size"), mediana_e_fondit=("Fondi_limit", "median"))
quarterly = by_date.resample("QS").agg(procedura=("Fondi_limit", "size"), mediana_e_fondit=("Fondi_limit", "median"))
yearly = by_date.resample("YS").agg(procedura=("Fondi_limit", "size"), mediana_e_fondit=("Fondi_limit", "median"))

display(monthly.head(12))
pd.DataFrame({
    "Numri i vlerave": [len(by_date), len(monthly), len(quarterly), len(yearly)],
    "Procedura mesatarisht në periudhë": [1, monthly["procedura"].mean(), quarterly["procedura"].mean(),
                                          yearly["procedura"].mean()],
}, index=["Procedurat", "Agregimi mujor", "Agregimi tremujor", "Agregimi vjetor"]).round(1)

## 8. Mostrimi

Mostra merr 10% të procedurave. U krahasuan tri metoda; zgjidhet mostra **e shtresëzuar sipas vitit dhe afërsisë me zgjedhjet**, që ruan saktësisht përqindjen e çdo viti dhe të periudhave zgjedhore. Këtu mostrat nxirren nga dataset-i i integruar.

In [ ]:
from src.common.sampling import compare_distributions
from src.time_integration.sampling import draw_samples

samples = draw_samples(integrated)
display(pd.DataFrame({name: [len(s), round(len(s) / len(integrated) * 100, 2)]
                      for name, s in samples.items()}, index=["rreshta", "% e dataset-it"]))

comparison = compare_distributions(integrated, samples, ["year", "afersia_me_zgjedhjet"])
comparison[comparison["variable"] == "afersia_me_zgjedhjet"]

In [ ]:
# Devijimi më i madh nga dataset-i i plotë, mesatarisht në 50 seed-e (pikë përqindjeje)
pd.read_csv("reports/time_integration/sampling_max_deviation.csv")

### Reservoir sampling

Kjo metodë zgjedh një mostër me madhësi fikse **k** nga një rrjedhë të dhënash, duke e lexuar çdo rresht vetëm një herë (p.sh. kur të dhënat nuk nxënë në memorie). Rreshti i *i*-të zëvendëson një element të mostrës me probabilitet *k / i*, kështu që në fund çdo rresht ka të njëjtën gjasë të jetë në mostër.

In [ ]:
import numpy as np

def reservoir_sample(n_rows, k, seed=42):
    rng = np.random.default_rng(seed)
    reservoir = list(range(k))           # k rreshtat e parë hyjnë direkt
    for i in range(k, n_rows):           # çdo rresht tjetër lexohet një herë
        j = rng.integers(0, i + 1)
        if j < k:                        # zëvendëson me probabilitet k / (i + 1)
            reservoir[j] = i
    return reservoir

k = round(len(integrated) * 0.10)
reservoir = integrated.iloc[reservoir_sample(len(integrated), k)]

# Krahasimi i përqindjes sipas vitit me dataset-in e plotë dhe me mostrat e tjera
pd.DataFrame({
    "Dataset-i i plotë": integrated["year"].value_counts(normalize=True),
    "Reservoir": reservoir["year"].value_counts(normalize=True),
    **{name: s["year"].value_counts(normalize=True) for name, s in samples.items()},
}).mul(100).round(2).sort_index()

Reservoir sampling sillet si mostra e rastësishme e thjeshtë: përqindjet luhaten pak rreth dataset-it të plotë. Mostra e shtresëzuar sipas vitit dhe zgjedhjeve i ruan saktësisht, prandaj ajo mbetet zgjedhja e projektit.

---
# c) Krijimi i vetive, diskretizimi, binarizimi, transformimi, zgjedhja e vetive dhe reduktimi

## 9. Krijimi i vetive

In [ ]:
from src.time_integration.features import features_summary as time_features
from src.values_winners.features import features_summary as value_features

display(value_features(integrated)[["feature", "step", "definition", "non_null"]])
time_features(integrated)[["feature", "group", "definition", "non_null"]]

In [ ]:
# Vetitë e konkurrencës (procedura dhe konkurrenca)
competition = ["cpv_count", "primary_cpv_division_name", "qualified_offer_share", "single_bidder",
               "no_qualified_offers", "competition_band", "is_framework_agreement",
               "has_cancellation_reason", "cancelled_without_reason"]
pd.DataFrame({"non_null": integrated[competition].notna().sum(), "dtype": integrated[competition].dtypes.astype(str)})

## 10. Diskretizimi

Leksioni krahason ndarjen me **gjerësi të barabartë** dhe me **frekuencë të barabartë**. Te shumat dhe kohëzgjatjet, gjerësia e barabartë fut pothuajse të gjitha procedurat në intervalin e parë, sepse disa vlera shumë të mëdha e zgjerojnë intervalin.

In [ ]:
from src.common.features import compare_discretizations

# Fondi limit: zgjidhet frekuenca e barabartë (kuartilet)
compare_discretizations(integrated["Fondi_limit"], 4)

In [ ]:
from src.time_integration.features import DURATION_BINS, DURATION_LABELS, discretization_comparison

# Kohëzgjatja: 53% e tenderave zgjasin 1 ditë, ndaj kuantilet përsëriten -> kufij të zgjedhur
discretization_comparison(integrated["kohezgjatja_e_tenderit"], 5, DURATION_BINS, DURATION_LABELS)

In [ ]:
# Numri i ofertave në 4 grupe
integrated["competition_band"].value_counts(dropna=False).rename("procedura").to_frame()

## 11. Binarizimi

In [ ]:
from src.time_integration.features import binary_summary

value_binaries = ["me_tvsh", "fitues_i_perbashket", "Fondi_limit_missing",
                  "Vlera_e_fituesit_missing", "Fituesi_missing"]
pd.concat([
    pd.DataFrame([{"feature": f, "ones": int(integrated[f].sum()),
                   "pct_ones": round(integrated[f].mean() * 100, 2)} for f in value_binaries]),
    binary_summary(integrated),
], ignore_index=True)

## 12. Transformimi

Shumat në lekë kanë asimetri shumë të madhe. Fillimisht aplikohet **log(1 + x)**, pastaj **min-max** (0–1) dhe **z-score** (mesatarja 0, devijimi 1). Min-max pa log nuk funksionon: pothuajse të gjitha vlerat grumbullohen afër 0. Kategoritë e procedurave kodohen me **one-hot** dhe **frequency encoding**.

In [ ]:
from src.values_winners.transformation import transform

_, transformation = transform(integrated)
transformation[["column", "transformation", "skew_before", "min_after", "max_after",
                "mean_after", "std_after", "skew_after"]]

In [ ]:
from src.procedure_competition.encoding import encode_categories

encoded, encoding = encode_categories(procedures)
encoding

## 13. Zgjedhja e nën-bashkësisë së vetive dhe korrelacioni

Për procedurat dhe konkurrencën, kolonat tekstuale dhe kategoritë e zëvendësuara nga encoding-u përjashtohen. Për PCA-në zgjidhen vetitë sasiore të të tri pjesëve; **korrelacioni** tregon vetitë që mbartin të njëjtin informacion (p.sh. ofertat e dorëzuara dhe të kualifikuara: 0.86).

In [ ]:
from src.procedure_competition.feature_selection import select_analysis_features

_, selection = select_analysis_features(encoded)
display(selection["status"].value_counts().to_frame())
selection[selection["status"] == "excluded"][["column", "reason"]]

In [ ]:
from src.values_winners.pca import FEATURES, feature_matrix

X = feature_matrix(integrated)  # log + z-score, vetëm rreshtat me të gjitha vetitë
print(f"{len(X):,} procedura, {X.shape[1]} veti")
X.corr().round(2)

## 14. Reduktimi i dimensionit (PCA)

In [ ]:
from src.common.pca import components_for, pca

explained, loadings, _ = pca(X)
print(f"Komponentë për 80% të variancës: {components_for(explained, 80)}, "
      f"për 90%: {components_for(explained, 90)}")
explained

In [ ]:
loadings[["PC1", "PC2", "PC3", "PC4", "PC5"]]

- **PC1** (29%): madhësia e procedurës kundrejt konkurrencës (fondi, vlera, kohëzgjatja + ; ofertat −).
- **PC4** (11%): koha në raport me zgjedhjet, një dimension pothuajse i pavarur nga të tjerët.

---
# Dataset-i përfundimtar

`prokurimi_final.csv` = dataset-i i integruar + konsistenca mes pjesëve + transformimi + 6 komponentët PCA (≥ 90% e variancës), pa kolonat konstante. Shpjegimi: `docs/final_dataset.md`.

**Konsistenca:** 27,707 procedura janë të anuluara, por kanë fitues. Asgjë nuk ndryshohet; shtohen `kontrate_e_lidhur` (ka fitues dhe nuk është e anuluar) dhe `anulim_kontradiktor` (e anuluar, por me kontratë të lidhur ose me arsyen "asnjë ofertë" ndërkohë që ka oferta). Trajtimi i tyre vendoset në Pjesën II.

In [ ]:
from src.final.final_dataset import build

final, tables = build(raw)
display(tables["final_summary"])
tables["final_consistency"]

In [ ]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
final.to_csv("data/processed/prokurimi_final.csv", index=False)
print(f"U ruajt data/processed/prokurimi_final.csv: {final.shape[0]:,} rreshta x {final.shape[1]} kolona")